# 01_text_preprocessing: Cleaning, Normalization, Stemming, and Subword Simulation
    
This notebook implements classical text preprocessing steps (Porter stemming and WordNet lemmatization) using NLTK on a scraped Wikipedia page corpus, and simulates a basic Byte-Pair Encoding (BPE) subword merge loop.


In [1]:
import nltk
import re
import requests
from bs4 import BeautifulSoup
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# 1. Download required NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# 2. Scrape Wikipedia NLP page
url = "https://en.wikipedia.org/wiki/Natural_language_processing"
resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(resp.content, "html.parser")
paragraphs = [p.get_text().strip() for p in soup.find_all("p") if len(p.get_text().strip()) > 80]
raw_text = paragraphs[1]
print("Raw Scraped Wikipedia Text snippet:", raw_text[:120], "...\n")

# 3. Basic regex cleaning
cleaned_text = re.sub(r"https?://\S+", "", raw_text)
cleaned_text = re.sub(r"[^\w\s]", "", cleaned_text).lower()
print("Cleaned Text:", cleaned_text[:120], "...\n")

# 4. Tokenize
tokens = word_tokenize(cleaned_text)[:15] # take a subset of tokens
print("Tokens:", tokens)

# 5. Stemming vs Lemmatization
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stemmed = [stemmer.stem(t) for t in tokens]
lemmatized = [lemmatizer.lemmatize(t, pos='v') for t in tokens]

print("\nLexical Reduction Comparison:")
print(f"{'Original':<15} | {'Stemmed':<15} | {'Lemmatized':<15}")
print("-" * 51)
for o, s, l in zip(tokens, stemmed, lemmatized):
    print(f"{o:<15} | {s:<15} | {l:<15}")


Raw Scraped Wikipedia Text snippet: Major processing tasks in an NLP system include: speech recognition, text classification, natural language understanding ...

Cleaned Text: major processing tasks in an nlp system include speech recognition text classification natural language understanding an ...

Tokens: ['major', 'processing', 'tasks', 'in', 'an', 'nlp', 'system', 'include', 'speech', 'recognition', 'text', 'classification', 'natural', 'language', 'understanding']



Lexical Reduction Comparison:
Original        | Stemmed         | Lemmatized     
---------------------------------------------------
major           | major           | major          
processing      | process         | process        
tasks           | task            | task           
in              | in              | in             
an              | an              | an             
nlp             | nlp             | nlp            
system          | system          | system         
include         | includ          | include        
speech          | speech          | speech         
recognition     | recognit        | recognition    
text            | text            | text           
classification  | classif         | classification 
natural         | natur           | natural        
language        | languag         | language       
understanding   | understand      | understand     


## Byte-Pair Encoding (BPE) Simulation
Let's simulate a basic bottom-up BPE tokenizer training merge loop on a tiny vocabulary extracted from the scraped text.


In [2]:
from collections import Counter, defaultdict

# BPE training corpus from scraped Wikipedia tokens
sample_text = "natural language processing language processing pipeline"
words = sample_text.split()
corpus = Counter([" ".join(list(w)) + " _" for w in words])

print("Initial split corpus counts:")
for w, freq in corpus.items():
    print(f"  {w}: {freq}")

def get_stats(corpus):
    pairs = defaultdict(int)
    for word, freq in corpus.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def merge_vocab(pair, corpus):
    new_corpus = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in corpus:
        w_new = p.sub(''.join(pair), word)
        new_corpus[w_new] = corpus[word]
    return new_corpus

# Run 5 BPE merge iterations
vocab = set("n a t u r l g e p o c s i d _".split())
print("\nInitial Vocab:", sorted(vocab))

for i in range(5):
    pairs = get_stats(corpus)
    if not pairs:
        break
    best_pair = max(pairs, key=pairs.get)
    corpus = merge_vocab(best_pair, corpus)
    merged_token = ''.join(best_pair)
    vocab.add(merged_token)
    print(f"\nIteration {i+1}: Merging {best_pair} (frequency={pairs[best_pair]})")
    print("Updated Corpus State:", corpus)

print("\nFinal BPE Vocab:", sorted(vocab))


Initial split corpus counts:
  n a t u r a l _: 1
  l a n g u a g e _: 2
  p r o c e s s i n g _: 2
  p i p e l i n e _: 1

Initial Vocab: ['_', 'a', 'c', 'd', 'e', 'g', 'i', 'l', 'n', 'o', 'p', 'r', 's', 't', 'u']

Iteration 1: Merging ('n', 'g') (frequency=4)
Updated Corpus State: {'n a t u r a l _': 1, 'l a ng u a g e _': 2, 'p r o c e s s i ng _': 2, 'p i p e l i n e _': 1}

Iteration 2: Merging ('e', '_') (frequency=3)
Updated Corpus State: {'n a t u r a l _': 1, 'l a ng u a g e_': 2, 'p r o c e s s i ng _': 2, 'p i p e l i n e_': 1}

Iteration 3: Merging ('l', 'a') (frequency=2)
Updated Corpus State: {'n a t u r a l _': 1, 'la ng u a g e_': 2, 'p r o c e s s i ng _': 2, 'p i p e l i n e_': 1}

Iteration 4: Merging ('la', 'ng') (frequency=2)
Updated Corpus State: {'n a t u r a l _': 1, 'lang u a g e_': 2, 'p r o c e s s i ng _': 2, 'p i p e l i n e_': 1}

Iteration 5: Merging ('lang', 'u') (frequency=2)
Updated Corpus State: {'n a t u r a l _': 1, 'langu a g e_': 2, 'p r o c e s s

### Output Explanation
- **Lexical Reduction**: Stemming cuts suffixes heuristically (e.g. `"processing"` $ightarrow$ `"process"`), whereas lemmatization resolves tokens to morphological base forms using grammatical tagging lookup tables (e.g. `"processing"` $ightarrow$ `"process"`).
- **BPE merges**: The simulator groups adjacent characters (like `(p, r)` then `(pr, o)`) based on statistical counts to form vocabulary subwords.
